<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/InfiniSplat_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🌌 InfiniSplat — Single-Image 3D Gaussian Reconstruction (SIGGRAPH Asia 2026, Apache 2.0)

A Colab port of [InfiniSplat](https://github.com/zju3dv/InfiniSplat) (Zhejiang University / PLUS-WAVE, SIGGRAPH Asia 2026 Journal Track). InfiniSplat reconstructs a 3D Gaussian Splat scene from a **single RGB image** using a ViT-L/16 backbone + implicit Gaussian decoder.

## What it does

Upload one image → get ~1.5M 3D Gaussians as a `.ply` file (compatible with SuperSplat / PlayCanvas / gsplat.js). Optionally render a novel-view orbit video.

Two modes:
- **RGB-only** — monocular 3D Gaussian reconstruction from a single image
- **Depth-sensor-guided** — RGB + depth map → higher-accuracy 3DGS (requires a depth file)

```
1 image (RGB)  →  ViT-L/16 encoder  →  1.5M Gaussians  →  .ply + orbit video
                 (+ depth optional)     (floater-filtered)
```

**Best for:** indoor scenes (trained on HyperSim). Outdoor scenes may produce artifacts. Occluded regions are hallucinated, not measured — the model cannot see behind objects.

## Quick start

1. **Runtime → Change runtime type → GPU** (T4, L4, or A100 — any works).
2. Run **STEP 1** — installs torch 2.9.1+cu128 + pip deps + clones repo + downloads checkpoints (~3 GB). First run: ~5-10 min.
3. Run **STEP 2** — imports + lazy model loader.
4. Run **STEP 3** — opens the Gradio UI. Upload an image, click **Reconstruct**.
5. **STEP 4** is keep-alive, **STEP 5** is a quick test, **STEP 6** is batch processing.

## Outputs

```
output.ply     # INRIA-style 3DGS (SuperSplat / PlayCanvas compatible)
output.mp4     # Novel-view orbit video (60 frames @ 10 fps, optional)
```

## Requirements

* **GPU**: T4 (15 GB) is sufficient. L4 / A100 are faster but not required.
* **Disk**: ~3 GB for checkpoints (cached on Drive after first run).
* **First-run setup**: ~5-10 min (pip installs + checkpoint download).
* **Subsequent runs**: ~15s (model load from cache).
* **No custom CUDA builds** — all deps are pre-built pip wheels.

## License

Apache License 2.0. Fully compatible with our suite.

## Companion notebooks

- **MapAnything_Colab** — Apache 2.0, universal 3DGS-from-images
- **NoPoSplat_Colab** — MIT, 2-3 photos → 3DGS in ~10 s
- **TripoSplat_Colab** — MIT, text/image → 3DGS
- **GaussianGPT_Colab** — MIT, autoregressive 3D scene generation (no input needed)
- **SplatTransform_Colab** — 3DGS format converter + voxel collision mesh


In [ ]:
#@title STEP 1 — Install deps, clone repo, download checkpoints
"""
• Pins torch to 2.9.1+cu128 (matches InfiniSplat's requirements)
• Installs all pip deps (hydra, timm, xformers, gsplat optional, etc.)
• Clones the InfiniSplat repo
• Downloads checkpoints from HuggingFace (~3 GB RGB, ~2.5 GB lidar)
• No custom CUDA builds — all deps are pre-built wheels
"""
import os, sys, time, subprocess, shutil, pathlib

print('='*72)
print('InfiniSplat — Install + Setup')
print('='*72)
try:
    import torch
    print(f'  Python : {sys.version.split()[0]}')
    print(f'  torch  : {torch.__version__}  CUDA: {torch.version.cuda}')
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f'  GPU    : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
    else:
        print('  WARNING: no GPU detected — inference will be very slow')
except ImportError:
    print('  torch not yet installed')
print()

CONNECT_GOOGLE_DRIVE = True  #@param {type:'boolean'}
if CONNECT_GOOGLE_DRIVE:
    drive_root = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/InfiniSplat')
    drive_root.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(drive_root / 'huggingface')
    print(f'  Drive cache  : {drive_root}')
else:
    drive_root = pathlib.Path('/content/_infinisplat_cache')
    drive_root.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(drive_root / 'huggingface')
    print(f'  Local cache  : {drive_root}')

OUT_DIR = drive_root / 'infinisplat_out'
OUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_ROOT = pathlib.Path('/content/infinisplat_work')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = WORK_ROOT / 'InfiniSplat'
CKPT_DIR = REPO_DIR / 'checkpoints'

t_total = time.time()

# 1. Pin torch to 2.9.1+cu128 ───────────────────────────────────────────
TARGET_TORCH = '2.9.1'
if not torch.__version__.startswith(TARGET_TORCH):
    print(f'\n[1/4] Pinning torch to {TARGET_TORCH}+cu128 ...')
    t0 = time.time()
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        '--disable-pip-version-check', '--no-input',
        f'torch=={TARGET_TORCH}+cu128',
        'torchvision==0.24.1+cu128',
        '--index-url', 'https://download.pytorch.org/whl/cu128',
        '--force-reinstall',
    ], check=False)
    print(f'  torch pinned in {time.time()-t0:.1f}s')
    import importlib
    importlib.reload(importlib.import_module('torch'))
    import torch
    print(f'  torch now    : {torch.__version__}  (CUDA {torch.version.cuda})')
else:
    print(f'\n[1/4] torch {torch.__version__} already matches {TARGET_TORCH} — skipping')

# 2. Clone InfiniSplat repo ─────────────────────────────────────────────
print('\n[2/4] Cloning InfiniSplat repo ...')
t0 = time.time()
if REPO_DIR.exists() and (REPO_DIR / '.git').exists():
    print('  Already cloned — pulling latest')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--quiet', '--ff-only'],
                   capture_output=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR, ignore_errors=True)
    subprocess.run(['git', 'clone', '--quiet', '--depth=1',
                     'https://github.com/zju3dv/InfiniSplat.git', str(REPO_DIR)],
                   check=True)
sys.path.insert(0, str(REPO_DIR))
print(f'  Cloned in {time.time()-t0:.1f}s')

# 3. Install pip deps ────────────────────────────────────────────────────
print('\n[3/4] Installing pip deps ...')
t0 = time.time()
EXTRA_PKGS = [
    'hydra-core', 'omegaconf', 'dacite', 'termcolor', 'rich', 'tqdm',
    'numpy<2.0', 'scipy', 'pandas', 'h5py', 'einops>=0.4.1', 'jaxtyping',
    'opencv-python-headless', 'scikit-learn', 'Pillow',
    'imageio[ffmpeg]', 'matplotlib', 'plyfile', 'torchmetrics',
    'timm', 'regex', 'huggingface-hub', 'gradio>=5.49.1,<7',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + EXTRA_PKGS, check=False)
# xformers (optional but recommended for ViT-L attention)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'xformers', '--index-url', 'https://download.pytorch.org/whl/cu128',
], check=False)
print(f'  Pip deps installed in {time.time()-t0:.1f}s')

# 4. Download checkpoints ────────────────────────────────────────────────
print('\n[4/4] Downloading checkpoints ...')
t0 = time.time()
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='PLUS-WAVE/InfiniSplat',
    allow_patterns=['checkpoints/*.ckpt'],
    local_dir=str(REPO_DIR),
    cache_dir=os.environ.get('HF_HOME'),
)
# Verify checkpoints
for name, expected_min_size in [
    ('infinisplat_rgb.ckpt', 2_000_000_000),
    ('infinisplat_lidar.ckpt', 1_500_000_000),
]:
    p = CKPT_DIR / name
    if p.exists():
        sz = p.stat().st_size / 1024**3
        print(f'  {name}: {sz:.2f} GB')
    else:
        print(f'  [WARN] {name}: not found')
print(f'  Checkpoints downloaded in {time.time()-t0:.1f}s')

elapsed = time.time() - t_total
print()
print('='*72)
print(f'STEP 1 complete in {elapsed/60:.1f} min')
print('='*72)
print(f'  Drive cache   : {drive_root}')
print(f'  Repo          : {REPO_DIR}')
print(f'  Checkpoints   : {CKPT_DIR}')
print()
print('Next: run STEP 2 (imports + lazy model loader).')


In [ ]:
#@title STEP 2 — Imports, lazy model loader, inference + export helpers
"""
• Verifies all deps are importable
• Defines `load_infinisplat()` — the lazy loader that picks the right
  checkpoint based on mode (rgb or lidar), loads the encoder + decoder
• Defines `inference_single_image()` — a convenience wrapper that
  runs the full pipeline: preprocess → encode → filter floaters → save .ply
• Defines `render_orbit_video()` — renders a novel-view video using gsplat
"""
import os, sys, time, gc, pathlib, traceback
import torch
import numpy as np
from PIL import Image, ImageOps

print('='*72)
print('InfiniSplat — Imports + lazy loader')
print('='*72)

# --- Verify deps ────────────────────────────────────────────────────────
deps_ok = True
for mod_name in ['torch', 'hydra', 'omegaconf', 'timm', 'einops',
                  'plyfile', 'scipy', 'cv2', 'imageio']:
    try:
        __import__(mod_name)
    except ImportError as e:
        print(f'  [FAIL] {mod_name}: {e}')
        deps_ok = False
if not deps_ok:
    raise RuntimeError('Missing dependencies — re-run STEP 1')

import timm
print(f'  torch  : {torch.__version__}  (CUDA {torch.version.cuda})')
print(f'  timm   : {timm.__version__}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'  GPU    : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
else:
    print('  WARNING: no GPU detected')
print()

# --- Add repo to path ──────────────────────────────────────────────────
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# --- Lazy model loader (singleton) ────────────────────────────────────
_ENCODER = None
_DECODER = None
_DEVICE = None
_MODE = None

def load_infinisplat(mode='rgb', verbose=True):
    """Load the InfiniSplat model from cached checkpoints.

    Args:
        mode: 'rgb' or 'lidar'
    Returns: (encoder, decoder, device)
    """
    global _ENCODER, _DECODER, _DEVICE, _MODE
    if _ENCODER is not None and _MODE == mode:
        return _ENCODER, _DECODER, _DEVICE

    from src.demo.infer_single_image import load_demo_config, load_demo_model

    experiment_name = 'infinisplat_hypersim_rgb' if mode == 'rgb' else 'infinisplat_hypersim_lidar'
    ckpt_name = 'infinisplat_rgb.ckpt' if mode == 'rgb' else 'infinisplat_lidar.ckpt'
    ckpt_path = CKPT_DIR / ckpt_name

    if not ckpt_path.exists():
        raise FileNotFoundError(f'Checkpoint not found: {ckpt_path}. Run STEP 1 first.')

    _DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    if verbose:
        print(f'  Loading config for {experiment_name} ...')
    cfg = load_demo_config(experiment_name)

    if verbose:
        print(f'  Loading model from {ckpt_path} ...')
    t0 = time.time()
    _ENCODER, _DECODER = load_demo_model(cfg, ckpt_path, _DEVICE)
    if verbose:
        n_enc = sum(p.numel() for p in _ENCODER.parameters())
        n_dec = sum(p.numel() for p in _DECODER.parameters())
        print(f'  Model loaded in {time.time()-t0:.1f}s (encoder {n_enc/1e6:.1f}M, decoder {n_dec/1e6:.1f}M params)')
    _MODE = mode
    return _ENCODER, _DECODER, _DEVICE

def free_infinisplat():
    """Unload the model and free GPU memory."""
    global _ENCODER, _DECODER, _DEVICE, _MODE
    if _ENCODER is not None:
        del _ENCODER
    if _DECODER is not None:
        del _DECODER
    _ENCODER = None
    _DECODER = None
    _DEVICE = None
    _MODE = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# --- Inference helper ───────────────────────────────────────────────────
def inference_single_image(image_path, mode='rgb', focal_mm=None,
                             floater_filter=True, verbose=True):
    """Run InfiniSplat inference on a single image.

    Args:
        image_path: Path to the input RGB image.
        mode: 'rgb' or 'lidar'.
        focal_mm: Optional focal length in mm (35mm equivalent). None = auto.
        floater_filter: Whether to remove spatial outliers.
        verbose: Print progress.

    Returns: (gaussians, focal_length_px, image_shape)
    """
    from src.demo.infer_single_image import (
        load_demo_image_bundle,
        run_single_image_inference,
        filter_final_gaussian_floaters,
    )

    encoder, decoder, device = load_infinisplat(mode=mode, verbose=verbose)

    if verbose:
        print(f'  Loading image: {image_path}')
    image_path = pathlib.Path(image_path)
    focal_kwargs = {}
    if focal_mm is not None and focal_mm > 0:
        focal_kwargs['focal_length_mm'] = float(focal_mm)
    bundle = load_demo_image_bundle(image_path, **focal_kwargs)

    if verbose:
        print(f'  Running encoder inference ...')
    t0 = time.time()
    with torch.inference_mode():
        output = run_single_image_inference(
            encoder=encoder,
            image=bundle.inference_image,
            intrinsics_px=bundle.inference_intrinsics.intrinsics_px,
            device=device,
        )
    if verbose:
        print(f'  Encoder done in {time.time()-t0:.1f}s')

    from src.utils.gaussians import Gaussians3D
    gaussians = Gaussians3D(
        mean_vectors=output['mean_vectors'],
        singular_values=output['singular_values'],
        quaternions=output['quaternions'],
        colors=output['colors'],
        opacities=output['opacities'],
        covariances=output.get('covariances'),
    )

    if floater_filter:
        if verbose:
            print(f'  Filtering floaters (kNN outlier removal) ...')
        t0 = time.time()
        with torch.inference_mode():
            gaussians = filter_final_gaussian_floaters(gaussians)
        if verbose:
            n_before = output['mean_vectors'].shape[1]
            n_after = gaussians.mean_vectors.shape[1]
            print(f'  Filtered {n_before - n_after:,} floaters in {time.time()-t0:.1f}s ({n_after:,} remaining)')

    focal_px = float(bundle.inference_intrinsics.focal_length_px)
    image_shape = bundle.original_image_shape
    return gaussians, focal_px, image_shape

# --- PLY export helper ─────────────────────────────────────────────────
def save_ply_file(gaussians, focal_px, image_shape, output_path):
    """Save Gaussians to an INRIA-style 3DGS .ply file."""
    from src.utils.gaussians import save_ply
    save_ply(gaussians, focal_px, image_shape, pathlib.Path(output_path))

# --- Orbit video rendering helper ──────────────────────────────────────
def render_orbit_video(gaussians, focal_px, image_shape, output_path,
                        fps=10, frames=60, verbose=True):
    """Render a novel-view orbit video using the gsplat decoder."""
    from src.demo.infer_single_image import render_novel_view_video_from_single_view
    decoder = _DECODER
    if decoder is None:
        if verbose:
            print('  [WARN] decoder not loaded, skipping video render')
        return None
    if verbose:
        print(f'  Rendering orbit video ({frames} frames @ {fps} fps) ...')
    t0 = time.time()
    render_novel_view_video_from_single_view(
        decoder=decoder,
        gaussians=gaussians,
        render_intrinsics_px=torch.tensor(
            [[focal_px, 0, image_shape[1]/2], [0, focal_px, image_shape[0]/2], [0, 0, 1]],
            dtype=torch.float32,
        ),
        render_image_shape=image_shape[::-1],
        output_path=pathlib.Path(output_path),
    )
    if verbose:
        print(f'  Video rendered in {time.time()-t0:.1f}s')
    return str(output_path)

# --- CUDA free helper ──────────────────────────────────────────────────
def free_cuda(verbose=False):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        if verbose:
            free, total = torch.cuda.mem_get_info()
            print(f'  [cuda] free={free/1024**3:.1f} GB / total={total/1024**3:.1f} GB')

print('STEP 2 complete — model loader + helpers ready.')
print('Next: run STEP 3 to open the Gradio UI.')


In [ ]:
#@title STEP 3 — Gradio UI (single image reconstruction + batch)
"""
• Two-column layout: left = input controls, right = output gallery
• Mode selector (rgb / lidar)
• Focal length override (optional)
• Floater filter toggle
• Render video toggle
• Download buttons for .ply and .mp4
• Concurrency limit = 2
"""
import os, sys, time, pathlib, traceback
import torch
import gradio as gr

CSS = """
#col-container   { margin: 0 auto; max-width: 1400px; }
#main-title h1   { font-size: 2.4em !important; }
"""

with gr.Blocks(css=CSS, delete_cache=(600, 600)) as demo:
    gr.Markdown(
        '# **InfiniSplat — Single-Image 3D Gaussian Reconstruction**',
        elem_id='main-title',
    )
    gr.Markdown(
        'Reconstruct 3D Gaussian splats from a single image. '
        'Powered by [InfiniSplat](https://github.com/zju3dv/InfiniSplat) '
        '(SIGGRAPH Asia 2026, Apache 2.0). Best for indoor scenes.'
    )

    with gr.Row(elem_id='col-container'):
        with gr.Column(scale=1, min_width=380):
            input_image = gr.Image(
                label='Input image',
                type='filepath',
                height=300,
            )
            mode = gr.Radio(
                choices=['rgb', 'lidar'],
                value='rgb',
                label='Mode',
                info='RGB-only (monocular) or lidar (depth-guided). Use lidar only if you have a matching depth file.',
            )
            with gr.Accordion('Advanced options', open=False):
                focal_mm = gr.Number(
                    value=0,
                    label='Focal length (mm, 35mm equiv)',
                    info='Override auto-detected focal length. 0 = auto from EXIF or default 30mm. Lower = wider FOV.',
                )
                floater_filter = gr.Checkbox(
                    value=True,
                    label='Filter floaters (kNN outlier removal)',
                    info='Remove spatial outliers from the generated Gaussians. Recommended.',
                )
                render_video = gr.Checkbox(
                    value=True,
                    label='Render orbit video',
                    info='Render a 60-frame orbit video using gsplat. Adds ~5-10s.',
                )
            btn_reconstruct = gr.Button('Reconstruct 3D scene', variant='primary')
            status_box = gr.Textbox(
                label='Status', interactive=False, lines=3,
                placeholder='Awaiting reconstruction...',
            )
            with gr.Accordion('Downloads', open=False):
                dl_ply = gr.File(label='Download .ply', file_count='single')
                dl_video = gr.File(label='Download .mp4', file_count='single')

        with gr.Column(scale=2):
            gr.Markdown('### Output')
            output_video = gr.Video(label='Orbit video', height=400)
            gr.Markdown(
                'The .ply file can be opened in [SuperSplat](https://supersplat.xyz) '
                'or PlayCanvas for full splat rendering.'
            )

    # --- Event wiring ---
    def reconstruct(img_path, md, focal, do_filter, do_video):
        try:
            if not img_path:
                raise gr.Error('Please upload an image first.')
            output_subdir = OUT_DIR / f'recon_{int(time.time())}'
            output_subdir.mkdir(parents=True, exist_ok=True)

            t0 = time.time()
            gaussians, focal_px, image_shape = inference_single_image(
                image_path=img_path,
                mode=md,
                focal_mm=float(focal) if focal and float(focal) > 0 else None,
                floater_filter=do_filter,
                verbose=True,
            )
            infer_time = time.time() - t0
            n_gaussians = gaussians.mean_vectors.shape[1]

            # Save .ply
            ply_path = output_subdir / 'scene.ply'
            save_ply_file(gaussians, focal_px, image_shape, ply_path)
            ply_size_mb = ply_path.stat().st_size / 1024 / 1024

            # Render video
            video_path = None
            if do_video:
                video_path = output_subdir / 'orbit.mp4'
                try:
                    render_orbit_video(gaussians, focal_px, image_shape,
                                        video_path, verbose=True)
                    video_path = str(video_path)
                except Exception as e:
                    print(f'  [WARN] video render failed: {e}')
                    video_path = None

            status = (
                f'Reconstruction complete in {infer_time:.1f}s. '
                f'{n_gaussians:,} Gaussians, .ply {ply_size_mb:.1f} MB. '
                f'Output: {output_subdir.name}'
            )
            free_cuda()
            return (
                gr.update(value=video_path) if video_path else gr.update(),
                gr.update(visible=True, value=status),
                gr.update(value=str(ply_path)),
                gr.update(value=video_path) if video_path else gr.update(),
            )
        except Exception as e:
            traceback.print_exc(limit=4)
            raise gr.Error(f'Reconstruction failed: {e}')

    btn_reconstruct.click(
        reconstruct,
        inputs=[input_image, mode, focal_mm, floater_filter, render_video],
        outputs=[output_video, status_box, dl_ply, dl_video],
    )

    def _welcome():
        return (
            'Upload an image, select a mode, then click "Reconstruct 3D scene". '
            'First run loads the model (~15s after STEP 1). Best for indoor scenes.'
        )
    demo.load(_welcome, inputs=None, outputs=[status_box])

# --- Queue + launch ────────────────────────────────────────────────────
demo.queue(concurrency_limit=2, max_size=8)
try:
    from IPython.display import clear_output
    clear_output()
    clear_output(wait=True)
except Exception:
    pass
demo.launch(share=False, server_name='0.0.0.0', server_port=7860, show_error=True, height=1100)


In [ ]:
#@title STEP 4 — Keep alive + session summary
"""Standard AEI-suite keep-alive cell."""
import os, sys, time, pathlib
import IPython
from IPython.display import display, Javascript

print('='*72)
print('Keep-alive timer started.')
print('='*72)

try:
    summary = {
        'cache_root'    : str(drive_root),
        'repo_dir'      : str(REPO_DIR),
        'ckpt_dir'      : str(CKPT_DIR),
        'out_dir'       : str(OUT_DIR),
        'torch'         : torch.__version__,
        'cuda'          : torch.version.cuda,
        'timm'          : timm.__version__,
        'gpu'           : None,
    }
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        summary['gpu'] = f'{p.name}  ({p.total_memory / (1024**3):.1f} GB)'
    print('\n  Session summary')
    print('  ' + '-'*68)
    for k, v in summary.items():
        print(f'  {k:18s}: {v}')
except Exception as e:
    print(f'  WARN: summary print failed: {e}')

display(Javascript('''
function ClickConnect() {
  console.log("Keeping Colab alive — ", new Date().toLocaleTimeString());
  document.querySelector("colab-connect-button")?.click();
}
setInterval(ClickConnect, 60000);
'''))
print('\n  Keep-alive timer registered (60 s interval).')


In [ ]:
#@title STEP 5 — Quick test (single image reconstruction)
"""
Stand-alone test.  Upload one image, reconstruct 3DGS, save .ply + video.
"""
import os, sys, time, pathlib
import torch
from IPython.display import display, FileLink

print('='*72)
print('InfiniSplat — single-image quick test')
print('='*72)

try:
    from google.colab import files
    HAVE_COLAB_FILES = True
except ImportError:
    HAVE_COLAB_FILES = False

if HAVE_COLAB_FILES:
    print('\n  Pick ONE image to test (PNG / JPG / WEBP).')
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit('No file uploaded.')
    IMAGE_PATH = pathlib.Path(next(iter(uploaded.keys())))
else:
    IMAGE_PATH = pathlib.Path('/content/test.jpg')

MODE = 'rgb'  #@param ['rgb', 'lidar']
FOCAL_MM = 0  #@param {type:"number"}
FILTER_FLOATERS = True  #@param {type:"boolean"}
RENDER_VIDEO = True  #@param {type:"boolean"}

output_subdir = OUT_DIR / f'quicktest_{int(time.time())}'
output_subdir.mkdir(parents=True, exist_ok=True)

print(f'  Image : {IMAGE_PATH}')
print(f'  Mode  : {MODE}')
print(f'  Output: {output_subdir}')
print()

t0 = time.time()
gaussians, focal_px, image_shape = inference_single_image(
    image_path=IMAGE_PATH,
    mode=MODE,
    focal_mm=float(FOCAL_MM) if FOCAL_MM > 0 else None,
    floater_filter=FILTER_FLOATERS,
)
infer_time = time.time() - t0
n_gaussians = gaussians.mean_vectors.shape[1]

ply_path = output_subdir / 'scene.ply'
save_ply_file(gaussians, focal_px, image_shape, ply_path)
ply_mb = ply_path.stat().st_size / 1024 / 1024

video_path = None
if RENDER_VIDEO:
    video_path = output_subdir / 'orbit.mp4'
    try:
        render_orbit_video(gaussians, focal_px, image_shape, video_path, verbose=True)
        video_path = str(video_path)
    except Exception as e:
        print(f'  [WARN] video render failed: {e}')
        video_path = None

print()
print('='*72)
print(f'Reconstruction complete in {infer_time:.1f}s')
print(f'  Gaussians  : {n_gaussians:,}')
print(f'  Focal (px) : {focal_px:.1f}')
print(f'  Image shape: {image_shape}')
print(f'  .ply       : {ply_path} ({ply_mb:.1f} MB)')
if video_path:
    print(f'  .mp4       : {video_path}')
print('='*72)

display(FileLink(str(ply_path), result_html_prefix='Download .ply: '))
if video_path:
    display(FileLink(video_path, result_html_prefix='Download .mp4: '))

free_cuda()
print('\nSTEP 5 complete. Open the Gradio UI (STEP 3) for the full experience.')


In [ ]:
#@title STEP 6 — Batch process a folder of images
"""
Batch processor.  Point it at a folder of images and generate one .ply
(+ optional .mp4) per image.  The model is loaded once and reused.

A progress log is written to batch_log.jsonl for resume after disconnect.
"""
import os, sys, time, json, pathlib, traceback
import torch
from IPython.display import display, FileLink

print('='*72)
print('InfiniSplat — Batch processing')
print('='*72)

BATCH_INPUT_FOLDER = '/content/images_in'  #@param {type:"string"}
MODE = 'rgb'  #@param ['rgb', 'lidar']
FOCAL_MM = 0  #@param {type:"number"}
FILTER_FLOATERS = True  #@param {type:"boolean"}
RENDER_VIDEO = True  #@param {type:"boolean"}
MAX_ITEMS = 0  #@param {type:"integer"}

batch_input = pathlib.Path(BATCH_INPUT_FOLDER)
if not batch_input.exists():
    batch_input.mkdir(parents=True, exist_ok=True)
    raise SystemExit(
        f'[Batch] Input folder {batch_input} did not exist (now created). '
        f'Upload images to that folder and re-run this cell.'
    )

IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.webp', '.bmp'}
images = sorted([p for p in batch_input.iterdir()
                 if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
if not images:
    raise SystemExit(f'[Batch] No images found in {batch_input}')
if MAX_ITEMS > 0:
    images = images[:MAX_ITEMS]
print(f'  Found {len(images)} image(s) in {batch_input}')

output_subdir = OUT_DIR / f'batch_{int(time.time())}'
output_subdir.mkdir(parents=True, exist_ok=True)
batch_log = output_subdir / 'batch_log.jsonl'

print(f'  Mode     : {MODE}')
print(f'  Output   : {output_subdir}')
print()

results = []
batch_start = time.time()
log_f = open(batch_log, 'a', buffering=1)

for i, img_path in enumerate(images, 1):
    stem = img_path.stem
    print(f'  [{i:03d}/{len(images)}] {img_path.name} ...', end='', flush=True)
    t0 = time.time()
    try:
        case_dir = output_subdir / stem
        case_dir.mkdir(parents=True, exist_ok=True)
        gaussians, focal_px, image_shape = inference_single_image(
            image_path=img_path,
            mode=MODE,
            focal_mm=float(FOCAL_MM) if FOCAL_MM > 0 else None,
            floater_filter=FILTER_FLOATERS,
            verbose=False,
        )
        n_gaussians = gaussians.mean_vectors.shape[1]
        ply_path = case_dir / f'{stem}.ply'
        save_ply_file(gaussians, focal_px, image_shape, ply_path)
        video_path = None
        if RENDER_VIDEO:
            video_path = case_dir / f'{stem}.mp4'
            try:
                render_orbit_video(gaussians, focal_px, image_shape,
                                    video_path, verbose=False)
                video_path = str(video_path)
            except Exception as e:
                print(f' [video fail: {e}]', end='')
                video_path = None
        elapsed = time.time() - t0
        print(f' OK ({elapsed:.0f}s, {n_gaussians:,} gaussians)')
        results.append(('ok', stem, str(ply_path)))
        log_f.write(json.dumps({
            'idx': i, 'stem': stem, 'status': 'ok',
            'elapsed_s': elapsed, 'n_gaussians': n_gaussians,
            'ply': str(ply_path), 'video': video_path,
        }) + '\n')
    except Exception as e:
        elapsed = time.time() - t0
        print(f' FAIL ({elapsed:.0f}s): {e}')
        traceback.print_exc(limit=2)
        results.append(('error', stem, str(e)))
        log_f.write(json.dumps({
            'idx': i, 'stem': stem, 'status': 'error',
            'error': str(e), 'elapsed_s': elapsed,
        }) + '\n')
    free_cuda()

log_f.close()
total_elapsed = time.time() - batch_start
n_ok = sum(1 for r in results if r[0] == 'ok')
n_err = sum(1 for r in results if r[0] == 'error')

print()
print('='*72)
print(f'Batch complete: {n_ok} ok / {n_err} errors in {total_elapsed:.0f}s')
print(f'  Output: {output_subdir}')
print(f'  Log:   {batch_log}')
print('='*72)

for f in sorted(output_subdir.rglob('*.ply')):
    sz = f.stat().st_size / 1024 / 1024
    print(f'  {f.relative_to(output_subdir)}  ({sz:.1f} MB)')

if n_ok > 0:
    print(f'\n  Tip: zip with `!cd {output_subdir} && zip -r batch.zip .`')
    print(f'  Tip: open .ply files in [SuperSplat](https://supersplat.xyz)')


In [ ]:
#@title STEP 7 — Help / Format Reference / Pipeline Overview
"""Reference material for troubleshooting."""
import IPython.display as idd

help_md = '''# InfiniSplat — Help, Format Reference, and Pipeline Overview

## What this notebook does

Reconstructs a 3D Gaussian Splat scene from a **single RGB image** using [InfiniSplat](https://github.com/zju3dv/InfiniSplat) (SIGGRAPH Asia 2026, Apache 2.0). The output is a `.ply` file with ~1.5M Gaussians, compatible with SuperSplat / PlayCanvas / gsplat.js.

## Pipeline

```
1 image (RGB)
   ↓
ViT-L/16 encoder (image → latent features)
   ↓
Implicit Gaussian decoder (features → 1.5M Gaussians)
   ↓
Floater filter (kNN outlier removal)
   ↓
.ply export (INRIA-style 3DGS)
   ↓
Optional: orbit video render (gsplat, 60 frames @ 10 fps)
```

## Compute

| GPU | VRAM | Speed (per image) | Notes |
|-----|------|-------------------|-------|
| A100 | 40 GB | ~5-10 s | Fast. Overkill. |
| L4 | 22 GB | ~10-15 s | Good. Recommended. |
| T4 | 15 GB | ~15-30 s | Works fine. Cheapest. |
| CPU | — | ~5-10 min | Very slow. Not recommended. |

## Known issues

1. **Occluded regions are hallucinated.** The model cannot see behind objects. It infers 3D from a single 2D image, so back-facing surfaces are guesses, not measurements.
2. **Best for indoor scenes.** The model was trained on HyperSim (synthetic indoor scenes). Outdoor scenes may produce artifacts.
3. **Depth-guided mode requires a depth file.** The `lidar` mode needs a matching `.npy` / `.npz` / `.h5` / `.exr` depth file with the same filename stem as the RGB image.
4. **Focal length matters.** Wrong focal length → distorted reconstruction. Use EXIF data (auto-detected) or override with `Focal length (mm)`.
5. **Floaters.** Small isolated Gaussians far from the main scene. The floater filter (kNN outlier removal) cleans these up.

## When to use InfiniSplat vs the other 3D notebooks

| | InfiniSplat (this) | MapAnything | NoPoSplat | TripoSplat | GaussianGPT |
|--|-------------------|-------------|-----------|------------|--------------|
| Input | 1 image | 1-100 images | 2-3 photos | text/image | None |
| Method | ViT + implicit decoder | Feed-forward | Feed-forward | TripoSR + gsplat | Autoregressive GPT |
| Output | 3DGS .ply + video | 3DGS .ply + GLB | 3DGS .ply | 3DGS .ply/.splat | 3DGS .ply + GIF |
| Speed | 10-30s | ~10s | ~10s | ~30s | 5-15s |
| License | Apache 2.0 | Apache 2.0 | MIT | MIT | MIT |
| Use for | Single-image 3DGS | Multi-image 3DGS | Pose-free 3DGS | Text/image 3DGS | Generative 3DGS |

## Format reference

| Format | What it is | Where to use it |
|--------|------------|-----------------|
| `.ply` | INRIA-style 3DGS (x,y,z, SH, opacity, scale, rotation) | SuperSplat, PlayCanvas, gsplat.js, Three.js |
| `.mp4` | Orbit video (60 frames @ 10 fps) | Browser preview, social media |

## See also

- **MapAnything_Colab.ipynb** — multi-image 3DGS, Apache 2.0
- **NoPoSplat_Colab.ipynb** — 2-3 photos → 3DGS, MIT
- **TripoSplat_Colab.ipynb** — text/image → 3DGS, MIT
- **GaussianGPT_Colab.ipynb** — generative 3DGS (no input), MIT
- **SplatTransform_Colab.ipynb** — 3DGS format converter
- **TextureMapPrep_Colab.ipynb** — PBR texture maps

## Citation

```bibtex
@article{infinisplat2026,
  title  = {InfiniSplat: Implicit Gaussian Decoding for Large-Baseline Monocular View Synthesis},
  author = {Wang, Jiawei and Yu, Hao and Hu, Yongzhen and Yang, Xinyi and Ni, Tao and Zhan, Xin and Chen, Junbo and Zhou, Xiaowei and Hu, Ruizhen and Peng, Sida},
  journal = {SIGGRAPH Asia 2026 (Journal Track)},
  year   = {2026},
}
```
'''

print(help_md)
